# m02 · Chu trình công suất khí: các phép tính kiểm chứng

Sổ tay này tính lại bằng Python các ví dụ chính của module m02 (Slide Ch.2; Tutorials I; Cengel Ch.9; El-Sayed Ví dụ 4.2). Mỗi mục có ba phần: câu hỏi mà phương pháp trả lời → mã → diễn giải số in thật. Chạy từ trên xuống (cần numpy, scipy). Bảng A-17 được nhúng sẵn trong ô mã đầu tiên.

## 1. Otto với nhiệt dung thay đổi (Tutorials I #1)

**Câu hỏi:** chu trình Otto r = 8, 100 kPa, 290 K, q_in = 800 kJ/kg: T₃, P₃, w_net, η, MEP tính bằng bảng A-17 khác công thức khí lạnh bao nhiêu?

In [1]:
import math, numpy as np
from scipy.integrate import quad
from scipy.optimize import brentq, minimize_scalar
R = 0.287                                    # kJ/(kg K), Table A-1
# Table A-17 (air, ideal gas), rows read from the page image: T [K], h [kJ/kg], P_r, u [kJ/kg], v_r, s0
A17 = np.array([[200.0, 199.97, 0.3363, 142.56, 1707.0, 1.29559], [210.0, 209.97, 0.3987, 149.69, 1512.0, 1.34444], [220.0, 219.97, 0.469, 156.82, 1346.0, 1.39105], [230.0, 230.02, 0.5477, 164.0, 1205.0, 1.43557], [240.0, 240.02, 0.6355, 171.13, 1084.0, 1.47824], [250.0, 250.05, 0.7329, 178.28, 979.0, 1.51917], [260.0, 260.09, 0.8405, 185.45, 887.8, 1.55848], [270.0, 270.11, 0.959, 192.6, 808.0, 1.59634], [280.0, 280.13, 1.0889, 199.75, 738.0, 1.63279], [285.0, 285.14, 1.1584, 203.33, 706.1, 1.65055], [290.0, 290.16, 1.2311, 206.91, 676.1, 1.66802], [295.0, 295.17, 1.3068, 210.49, 647.9, 1.68515], [298.0, 298.18, 1.3543, 212.64, 631.9, 1.69528], [300.0, 300.19, 1.386, 214.07, 621.2, 1.70203], [305.0, 305.22, 1.4686, 217.67, 596.0, 1.71865], [310.0, 310.24, 1.5546, 221.25, 572.3, 1.73498], [315.0, 315.27, 1.6442, 224.85, 549.8, 1.75106], [320.0, 320.29, 1.7375, 228.42, 528.6, 1.7669], [325.0, 325.31, 1.8345, 232.02, 508.4, 1.78249], [330.0, 330.34, 1.9352, 235.61, 489.4, 1.79783], [340.0, 340.42, 2.149, 242.82, 454.1, 1.8279], [350.0, 350.49, 2.379, 250.02, 422.2, 1.85708], [360.0, 360.58, 2.626, 257.24, 393.4, 1.88543], [370.0, 370.67, 2.892, 264.46, 367.2, 1.91313], [380.0, 380.77, 3.176, 271.69, 343.4, 1.94001], [390.0, 390.88, 3.481, 278.93, 321.5, 1.96633], [400.0, 400.98, 3.806, 286.16, 301.6, 1.99194], [410.0, 411.12, 4.153, 293.43, 283.3, 2.01699], [420.0, 421.26, 4.522, 300.69, 266.6, 2.04142], [430.0, 431.43, 4.915, 307.99, 251.1, 2.06533], [440.0, 441.61, 5.332, 315.3, 236.8, 2.0887], [450.0, 451.8, 5.775, 322.62, 223.6, 2.11161], [460.0, 462.02, 6.245, 329.97, 211.4, 2.13407], [470.0, 472.24, 6.742, 337.32, 200.1, 2.15604], [480.0, 482.49, 7.268, 344.7, 189.5, 2.1776], [490.0, 492.74, 7.824, 352.08, 179.7, 2.19876], [500.0, 503.02, 8.411, 359.49, 170.6, 2.21952], [510.0, 513.32, 9.031, 366.92, 162.1, 2.23993], [520.0, 523.63, 9.684, 374.36, 154.1, 2.25997], [530.0, 533.98, 10.37, 381.84, 146.7, 2.27967], [540.0, 544.35, 11.1, 389.34, 139.7, 2.29906], [550.0, 555.74, 11.86, 396.86, 133.1, 2.31809], [560.0, 565.17, 12.66, 404.42, 127.0, 2.33685], [570.0, 575.59, 13.5, 411.97, 121.2, 2.35531], [580.0, 586.04, 14.38, 419.55, 115.7, 2.37348], [590.0, 596.52, 15.31, 427.15, 110.6, 2.3914], [600.0, 607.02, 16.28, 434.78, 105.8, 2.40902], [610.0, 617.53, 17.3, 442.42, 101.2, 2.42644], [620.0, 628.07, 18.36, 450.09, 96.92, 2.44356], [630.0, 638.63, 19.84, 457.78, 92.84, 2.46048], [640.0, 649.22, 20.64, 465.5, 88.99, 2.47716], [650.0, 659.84, 21.86, 473.25, 85.34, 2.49364], [660.0, 670.47, 23.13, 481.01, 81.89, 2.50985], [670.0, 681.14, 24.46, 488.81, 78.61, 2.52589], [680.0, 691.82, 25.85, 496.62, 75.5, 2.54175], [690.0, 702.52, 27.29, 504.45, 72.56, 2.55731], [700.0, 713.27, 28.8, 512.33, 69.76, 2.57277], [710.0, 724.04, 30.38, 520.23, 67.07, 2.5881], [720.0, 734.82, 32.02, 528.14, 64.53, 2.60319], [730.0, 745.62, 33.72, 536.07, 62.13, 2.61803], [740.0, 756.44, 35.5, 544.02, 59.82, 2.6328], [750.0, 767.29, 37.35, 551.99, 57.63, 2.64737], [760.0, 778.18, 39.27, 560.01, 55.54, 2.66176], [780.0, 800.03, 43.35, 576.12, 51.64, 2.69013], [800.0, 821.95, 47.75, 592.3, 48.08, 2.71787], [820.0, 843.98, 52.59, 608.59, 44.84, 2.74504], [840.0, 866.08, 57.6, 624.95, 41.85, 2.7717], [860.0, 888.27, 63.09, 641.4, 39.12, 2.79783], [880.0, 910.56, 68.98, 657.95, 36.61, 2.82344], [900.0, 932.93, 75.29, 674.58, 34.31, 2.84856], [920.0, 955.38, 82.05, 691.28, 32.18, 2.87324], [940.0, 977.92, 89.28, 708.08, 30.22, 2.89748], [960.0, 1000.55, 97.0, 725.02, 28.4, 2.92128], [980.0, 1023.25, 105.2, 741.98, 26.73, 2.94468], [1000.0, 1046.04, 114.0, 758.94, 25.17, 2.9677], [1020.0, 1068.89, 123.4, 776.1, 23.72, 2.99034], [1040.0, 1091.85, 133.3, 793.36, 23.29, 3.0126], [1060.0, 1114.86, 143.9, 810.62, 21.14, 3.03449], [1080.0, 1137.89, 155.2, 827.88, 19.98, 3.05608], [1100.0, 1161.07, 167.1, 845.33, 18.896, 3.07732], [1120.0, 1184.28, 179.7, 862.79, 17.886, 3.09825], [1140.0, 1207.57, 193.1, 880.35, 16.946, 3.11883], [1160.0, 1230.92, 207.2, 897.91, 16.064, 3.13916], [1180.0, 1254.34, 222.2, 915.57, 15.241, 3.15916], [1200.0, 1277.79, 238.0, 933.33, 14.47, 3.17888], [1220.0, 1301.31, 254.7, 951.09, 13.747, 3.19834], [1240.0, 1324.93, 272.3, 968.95, 13.069, 3.21751], [1260.0, 1348.55, 290.8, 986.9, 12.435, 3.23638], [1280.0, 1372.24, 310.4, 1004.76, 11.835, 3.2551], [1300.0, 1395.97, 330.9, 1022.82, 11.275, 3.27345], [1320.0, 1419.76, 352.5, 1040.88, 10.747, 3.2916], [1340.0, 1443.6, 375.3, 1058.94, 10.247, 3.30959], [1360.0, 1467.49, 399.1, 1077.1, 9.78, 3.32724], [1380.0, 1491.44, 424.2, 1095.26, 9.337, 3.34474], [1400.0, 1515.42, 450.5, 1113.52, 8.919, 3.362], [1420.0, 1539.44, 478.0, 1131.77, 8.526, 3.37901], [1440.0, 1563.51, 506.9, 1150.13, 8.153, 3.39586], [1460.0, 1587.63, 537.1, 1168.49, 7.801, 3.41247], [1480.0, 1611.79, 568.8, 1186.95, 7.468, 3.42892], [1500.0, 1635.97, 601.9, 1205.41, 7.152, 3.44516], [1520.0, 1660.23, 636.5, 1223.87, 6.854, 3.4612], [1540.0, 1684.51, 672.8, 1242.43, 6.569, 3.47712], [1560.0, 1708.82, 710.5, 1260.99, 6.301, 3.49276], [1580.0, 1733.17, 750.0, 1279.65, 6.046, 3.50829], [1600.0, 1757.57, 791.2, 1298.3, 5.804, 3.52364], [1620.0, 1782.0, 834.1, 1316.96, 5.574, 3.53879], [1640.0, 1806.46, 878.9, 1335.72, 5.355, 3.55381], [1660.0, 1830.96, 925.6, 1354.48, 5.147, 3.56867], [1680.0, 1855.5, 974.2, 1373.24, 4.949, 3.58335], [1700.0, 1880.1, 1025.0, 1392.7, 4.761, 3.5979], [1750.0, 1941.6, 1161.0, 1439.8, 4.328, 3.6336], [1800.0, 2003.3, 1310.0, 1487.2, 3.994, 3.6684]])
COL = dict(T=0, h=1, Pr=2, u=3, vr=4, s0=5)
def tab(col, T):                             # value of a column at temperature T (linear interpolation, as a student does)
    return float(np.interp(T, A17[:, 0], A17[:, COL[col]]))
def inv(col, val):                           # temperature at which a column has the value val
    x = A17[:, COL[col]]; T = A17[:, 0]
    return float(np.interp(val, x, T)) if x[0] < x[-1] else float(np.interp(val, x[::-1], T[::-1]))
r, T1, P1, qin = 8.0, 290.0, 100.0, 800.0
u1, vr1 = tab('u', T1), tab('vr', T1)
T2 = inv('vr', vr1 / r); u2 = tab('u', T2)                      # isentropic compression: v_r2 = v_r1 / r
T3 = inv('u', u2 + qin); vr3 = tab('vr', T3)                    # constant volume: u3 = u2 + q_in
T4 = inv('vr', vr3 * r); u4 = tab('u', T4)                      # isentropic expansion: v_r4 = r v_r3
P3 = P1 * (T3 / T1) * r
qout = u4 - u1; wnet = qin - qout; v1 = R * T1 / P1
print(f"T2 = {T2:.1f} K, T3 = {T3:.1f} K, T4 = {T4:.1f} K, P3 = {P3:.0f} kPa")
print(f"q_out = {qout:.2f}, w_net = {wnet:.2f} kJ/kg, eta = {wnet/qin:.3f}, MEP = {wnet/(v1*(1-1/r)):.0f} kPa")
k, cv = 1.4, 0.718                                              # cold air
T3c = T1 * r**(k-1) + qin / cv
print(f"cold air: T3 = {T3c:.1f} K, eta = {1 - r**(1-k):.3f}")

T2 = 652.4 K, T3 = 1575.1 K, T4 = 795.6 K, P3 = 4345 kPa
q_out = 381.83, w_net = 418.17 kJ/kg, eta = 0.523, MEP = 574 kPa
cold air: T3 = 1780.5 K, eta = 0.565


**Đọc kết quả:** bảng cho T₃ = 1575.1 K, P₃ = 4345 kPa, w_net = 418.17 kJ/kg, η = 0.523, MEP = 574 kPa: đúng đáp số slide Tutorials I (1575,1 K; 4,345 MPa; 418,17; 52,3 %; 574 kPa). Khí lạnh cho T₃ = 1780.5 K và η = 0.565: c_v thật nhỏ hơn 0,718 ở nhiệt độ thấp và lớn hơn ở nhiệt độ cao nên hiệu suất thật thấp hơn.

## 2. Diesel khí lạnh: công thức đóng và tích phân ∮P dV (Tutorials I #6)

**Câu hỏi:** động cơ diesel r = 20, 95 kPa, 293 K, T_max = 2200 K: r_c, η, MEP; công thức hiệu suất có khớp công thức tích phân ∮P dV không?

In [2]:
import math
from scipy.integrate import quad
k, R, cp, cv = 1.4, 0.287, 1.005, 0.718
r, T1, P1, T3 = 20.0, 293.0, 95.0, 2200.0
T2 = T1 * r**(k-1); rc = T3 / T2
qin = cp * (T3 - T2); T4 = T3 * (rc / r)**(k-1); qout = cv * (T4 - T1); w = qin - qout; v1 = R * T1 / P1
eta_formula = 1 - r**(1-k) * (rc**k - 1) / (k * (rc - 1))
# route B: net work from the integral of P dv along the four processes
v2 = v1 / r; v3 = rc * v2; P2 = R * T2 / v2
w12 = quad(lambda v: P1 * v1**k / v**k, v1, v2)[0]               # isentropic compression (negative)
w23 = P2 * (v3 - v2)                                              # isobaric heat addition
w34 = quad(lambda v: P2 * v3**k / v**k, v3, v1)[0]                # isentropic expansion
wint = w12 + w23 + w34
print(f"T2 = {T2:.1f} K, r_c = {rc:.3f}, T4 = {T4:.1f} K")
print(f"eta (heats) = {w/qin:.4f}, eta (r_c formula) = {eta_formula:.4f}, eta (integral) = {wint/qin:.4f}")
print(f"w_net = {w:.1f} kJ/kg (heats) vs {wint:.1f} kJ/kg (integral); MEP = {w/(v1*(1-1/r)):.1f} kPa")

T2 = 971.1 K, r_c = 2.265, T4 = 920.6 K
eta (heats) = 0.6351, eta (r_c formula) = 0.6352, eta (integral) = 0.6349
w_net = 784.4 kJ/kg (heats) vs 784.1 kJ/kg (integral); MEP = 932.8 kPa


**Đọc kết quả:** r_c = 2.265; η = 0.6351 theo cả ba cách (hiệu nhiệt, công thức r_c, tích phân); khác biệt nhỏ ở công thực (784.4 kJ/kg so với tích phân) do R/(k−1) = 0,7175 chứ không phải 0,718 (làm tròn hằng số của sách). MEP = 932.8 kPa (sách 933).

## 3. Brayton lý tưởng: bảng A-17 và đa thức c_p(T) của bảng A-2c (Tutorials I)

**Câu hỏi:** r_p = 8, 300 K và 1300 K: T₂, T₄, r_bw, η tính bằng bảng A-17 so với tích phân c_p(T) của đa thức A-2c (đường độc lập), và so với khí lạnh.

In [3]:
import math, numpy as np
from scipy.integrate import quad
from scipy.optimize import brentq, minimize_scalar
R = 0.287                                    # kJ/(kg K), Table A-1
# Table A-17 (air, ideal gas), rows read from the page image: T [K], h [kJ/kg], P_r, u [kJ/kg], v_r, s0
A17 = np.array([[200.0, 199.97, 0.3363, 142.56, 1707.0, 1.29559], [210.0, 209.97, 0.3987, 149.69, 1512.0, 1.34444], [220.0, 219.97, 0.469, 156.82, 1346.0, 1.39105], [230.0, 230.02, 0.5477, 164.0, 1205.0, 1.43557], [240.0, 240.02, 0.6355, 171.13, 1084.0, 1.47824], [250.0, 250.05, 0.7329, 178.28, 979.0, 1.51917], [260.0, 260.09, 0.8405, 185.45, 887.8, 1.55848], [270.0, 270.11, 0.959, 192.6, 808.0, 1.59634], [280.0, 280.13, 1.0889, 199.75, 738.0, 1.63279], [285.0, 285.14, 1.1584, 203.33, 706.1, 1.65055], [290.0, 290.16, 1.2311, 206.91, 676.1, 1.66802], [295.0, 295.17, 1.3068, 210.49, 647.9, 1.68515], [298.0, 298.18, 1.3543, 212.64, 631.9, 1.69528], [300.0, 300.19, 1.386, 214.07, 621.2, 1.70203], [305.0, 305.22, 1.4686, 217.67, 596.0, 1.71865], [310.0, 310.24, 1.5546, 221.25, 572.3, 1.73498], [315.0, 315.27, 1.6442, 224.85, 549.8, 1.75106], [320.0, 320.29, 1.7375, 228.42, 528.6, 1.7669], [325.0, 325.31, 1.8345, 232.02, 508.4, 1.78249], [330.0, 330.34, 1.9352, 235.61, 489.4, 1.79783], [340.0, 340.42, 2.149, 242.82, 454.1, 1.8279], [350.0, 350.49, 2.379, 250.02, 422.2, 1.85708], [360.0, 360.58, 2.626, 257.24, 393.4, 1.88543], [370.0, 370.67, 2.892, 264.46, 367.2, 1.91313], [380.0, 380.77, 3.176, 271.69, 343.4, 1.94001], [390.0, 390.88, 3.481, 278.93, 321.5, 1.96633], [400.0, 400.98, 3.806, 286.16, 301.6, 1.99194], [410.0, 411.12, 4.153, 293.43, 283.3, 2.01699], [420.0, 421.26, 4.522, 300.69, 266.6, 2.04142], [430.0, 431.43, 4.915, 307.99, 251.1, 2.06533], [440.0, 441.61, 5.332, 315.3, 236.8, 2.0887], [450.0, 451.8, 5.775, 322.62, 223.6, 2.11161], [460.0, 462.02, 6.245, 329.97, 211.4, 2.13407], [470.0, 472.24, 6.742, 337.32, 200.1, 2.15604], [480.0, 482.49, 7.268, 344.7, 189.5, 2.1776], [490.0, 492.74, 7.824, 352.08, 179.7, 2.19876], [500.0, 503.02, 8.411, 359.49, 170.6, 2.21952], [510.0, 513.32, 9.031, 366.92, 162.1, 2.23993], [520.0, 523.63, 9.684, 374.36, 154.1, 2.25997], [530.0, 533.98, 10.37, 381.84, 146.7, 2.27967], [540.0, 544.35, 11.1, 389.34, 139.7, 2.29906], [550.0, 555.74, 11.86, 396.86, 133.1, 2.31809], [560.0, 565.17, 12.66, 404.42, 127.0, 2.33685], [570.0, 575.59, 13.5, 411.97, 121.2, 2.35531], [580.0, 586.04, 14.38, 419.55, 115.7, 2.37348], [590.0, 596.52, 15.31, 427.15, 110.6, 2.3914], [600.0, 607.02, 16.28, 434.78, 105.8, 2.40902], [610.0, 617.53, 17.3, 442.42, 101.2, 2.42644], [620.0, 628.07, 18.36, 450.09, 96.92, 2.44356], [630.0, 638.63, 19.84, 457.78, 92.84, 2.46048], [640.0, 649.22, 20.64, 465.5, 88.99, 2.47716], [650.0, 659.84, 21.86, 473.25, 85.34, 2.49364], [660.0, 670.47, 23.13, 481.01, 81.89, 2.50985], [670.0, 681.14, 24.46, 488.81, 78.61, 2.52589], [680.0, 691.82, 25.85, 496.62, 75.5, 2.54175], [690.0, 702.52, 27.29, 504.45, 72.56, 2.55731], [700.0, 713.27, 28.8, 512.33, 69.76, 2.57277], [710.0, 724.04, 30.38, 520.23, 67.07, 2.5881], [720.0, 734.82, 32.02, 528.14, 64.53, 2.60319], [730.0, 745.62, 33.72, 536.07, 62.13, 2.61803], [740.0, 756.44, 35.5, 544.02, 59.82, 2.6328], [750.0, 767.29, 37.35, 551.99, 57.63, 2.64737], [760.0, 778.18, 39.27, 560.01, 55.54, 2.66176], [780.0, 800.03, 43.35, 576.12, 51.64, 2.69013], [800.0, 821.95, 47.75, 592.3, 48.08, 2.71787], [820.0, 843.98, 52.59, 608.59, 44.84, 2.74504], [840.0, 866.08, 57.6, 624.95, 41.85, 2.7717], [860.0, 888.27, 63.09, 641.4, 39.12, 2.79783], [880.0, 910.56, 68.98, 657.95, 36.61, 2.82344], [900.0, 932.93, 75.29, 674.58, 34.31, 2.84856], [920.0, 955.38, 82.05, 691.28, 32.18, 2.87324], [940.0, 977.92, 89.28, 708.08, 30.22, 2.89748], [960.0, 1000.55, 97.0, 725.02, 28.4, 2.92128], [980.0, 1023.25, 105.2, 741.98, 26.73, 2.94468], [1000.0, 1046.04, 114.0, 758.94, 25.17, 2.9677], [1020.0, 1068.89, 123.4, 776.1, 23.72, 2.99034], [1040.0, 1091.85, 133.3, 793.36, 23.29, 3.0126], [1060.0, 1114.86, 143.9, 810.62, 21.14, 3.03449], [1080.0, 1137.89, 155.2, 827.88, 19.98, 3.05608], [1100.0, 1161.07, 167.1, 845.33, 18.896, 3.07732], [1120.0, 1184.28, 179.7, 862.79, 17.886, 3.09825], [1140.0, 1207.57, 193.1, 880.35, 16.946, 3.11883], [1160.0, 1230.92, 207.2, 897.91, 16.064, 3.13916], [1180.0, 1254.34, 222.2, 915.57, 15.241, 3.15916], [1200.0, 1277.79, 238.0, 933.33, 14.47, 3.17888], [1220.0, 1301.31, 254.7, 951.09, 13.747, 3.19834], [1240.0, 1324.93, 272.3, 968.95, 13.069, 3.21751], [1260.0, 1348.55, 290.8, 986.9, 12.435, 3.23638], [1280.0, 1372.24, 310.4, 1004.76, 11.835, 3.2551], [1300.0, 1395.97, 330.9, 1022.82, 11.275, 3.27345], [1320.0, 1419.76, 352.5, 1040.88, 10.747, 3.2916], [1340.0, 1443.6, 375.3, 1058.94, 10.247, 3.30959], [1360.0, 1467.49, 399.1, 1077.1, 9.78, 3.32724], [1380.0, 1491.44, 424.2, 1095.26, 9.337, 3.34474], [1400.0, 1515.42, 450.5, 1113.52, 8.919, 3.362], [1420.0, 1539.44, 478.0, 1131.77, 8.526, 3.37901], [1440.0, 1563.51, 506.9, 1150.13, 8.153, 3.39586], [1460.0, 1587.63, 537.1, 1168.49, 7.801, 3.41247], [1480.0, 1611.79, 568.8, 1186.95, 7.468, 3.42892], [1500.0, 1635.97, 601.9, 1205.41, 7.152, 3.44516], [1520.0, 1660.23, 636.5, 1223.87, 6.854, 3.4612], [1540.0, 1684.51, 672.8, 1242.43, 6.569, 3.47712], [1560.0, 1708.82, 710.5, 1260.99, 6.301, 3.49276], [1580.0, 1733.17, 750.0, 1279.65, 6.046, 3.50829], [1600.0, 1757.57, 791.2, 1298.3, 5.804, 3.52364], [1620.0, 1782.0, 834.1, 1316.96, 5.574, 3.53879], [1640.0, 1806.46, 878.9, 1335.72, 5.355, 3.55381], [1660.0, 1830.96, 925.6, 1354.48, 5.147, 3.56867], [1680.0, 1855.5, 974.2, 1373.24, 4.949, 3.58335], [1700.0, 1880.1, 1025.0, 1392.7, 4.761, 3.5979], [1750.0, 1941.6, 1161.0, 1439.8, 4.328, 3.6336], [1800.0, 2003.3, 1310.0, 1487.2, 3.994, 3.6684]])
COL = dict(T=0, h=1, Pr=2, u=3, vr=4, s0=5)
def tab(col, T):                             # value of a column at temperature T (linear interpolation, as a student does)
    return float(np.interp(T, A17[:, 0], A17[:, COL[col]]))
def inv(col, val):                           # temperature at which a column has the value val
    x = A17[:, COL[col]]; T = A17[:, 0]
    return float(np.interp(val, x, T)) if x[0] < x[-1] else float(np.interp(val, x[::-1], T[::-1]))
rp, T1, T3 = 8.0, 300.0, 1300.0
# route A: Table A-17
T2 = inv('Pr', rp * tab('Pr', T1)); T4 = inv('Pr', tab('Pr', T3) / rp)
wc = tab('h', T2) - tab('h', T1); wt = tab('h', T3) - tab('h', T4); qin = tab('h', T3) - tab('h', T2)
print(f"table A-17: T2 = {T2:.1f} K, T4 = {T4:.1f} K, w_comp = {wc:.2f}, w_turb = {wt:.2f}, r_bw = {wc/wt:.4f}, eta = {(wt-wc)/qin:.4f}")
# route B: cp(T) = (a + bT + cT^2 + dT^3)/M, Table A-2c (air, 273-1800 K); isentrope: int cp/T dT = R ln(rp)
a, b, c, d, M = 28.11, 0.001967, 4.802e-6, -1.966e-9, 28.97
cp = lambda T: (a + b*T + c*T**2 + d*T**3) / M
h = lambda T1_, T2_: quad(cp, T1_, T2_)[0]
phi = lambda T1_, T2_: quad(lambda T: cp(T)/T, T1_, T2_)[0]
T2p = brentq(lambda T: phi(T1, T) - R*math.log(rp), T1, 1000)
T4p = brentq(lambda T: phi(T, T3) - R*math.log(rp), 300, T3)
wcp, wtp, qinp = h(T1, T2p), h(T4p, T3), h(T2p, T3)
print(f"polynomial: T2 = {T2p:.1f} K, T4 = {T4p:.1f} K, w_comp = {wcp:.2f}, w_turb = {wtp:.2f}, r_bw = {wcp/wtp:.4f}, eta = {(wtp-wcp)/qinp:.4f}")
print(f"cold air k = 1.4: eta = {1 - rp**(-0.4/1.4):.4f}, r_bw = {T1*rp**(0.4/1.4)/T3:.4f}")

table A-17: T2 = 539.8 K, T4 = 770.3 K, w_comp = 243.99, w_turb = 606.58, r_bw = 0.4022, eta = 0.4257
polynomial: T2 = 538.1 K, T4 = 769.2 K, w_comp = 243.69, w_turb = 606.01, r_bw = 0.4021, eta = 0.4250
cold air k = 1.4: eta = 0.4480, r_bw = 0.4180


**Đọc kết quả:** bảng A-17 cho T₂ = 539.8 K, T₄ = 770.3 K, r_bw = 0.4022, η = 0.4257; đa thức A-2c cho cùng kết quả trong khoảng 1 %. Slide làm tròn T₂ = 540 K nên in r_bw = 0,403 thay cho 0.4022. Khí lạnh cho η = 0.4480, cao hơn khoảng 2 điểm %.

## 4. Brayton thực và hồi nhiệt (Cengel Ví dụ 9–6, 9–7)

**Câu hỏi:** thêm η_c = 80 %, η_t = 85 % rồi bộ hồi nhiệt ε = 80 %: r_bw, η, T₄ₐ, h₅ thay đổi thế nào?

In [4]:
import math, numpy as np
from scipy.integrate import quad
from scipy.optimize import brentq, minimize_scalar
R = 0.287                                    # kJ/(kg K), Table A-1
# Table A-17 (air, ideal gas), rows read from the page image: T [K], h [kJ/kg], P_r, u [kJ/kg], v_r, s0
A17 = np.array([[200.0, 199.97, 0.3363, 142.56, 1707.0, 1.29559], [210.0, 209.97, 0.3987, 149.69, 1512.0, 1.34444], [220.0, 219.97, 0.469, 156.82, 1346.0, 1.39105], [230.0, 230.02, 0.5477, 164.0, 1205.0, 1.43557], [240.0, 240.02, 0.6355, 171.13, 1084.0, 1.47824], [250.0, 250.05, 0.7329, 178.28, 979.0, 1.51917], [260.0, 260.09, 0.8405, 185.45, 887.8, 1.55848], [270.0, 270.11, 0.959, 192.6, 808.0, 1.59634], [280.0, 280.13, 1.0889, 199.75, 738.0, 1.63279], [285.0, 285.14, 1.1584, 203.33, 706.1, 1.65055], [290.0, 290.16, 1.2311, 206.91, 676.1, 1.66802], [295.0, 295.17, 1.3068, 210.49, 647.9, 1.68515], [298.0, 298.18, 1.3543, 212.64, 631.9, 1.69528], [300.0, 300.19, 1.386, 214.07, 621.2, 1.70203], [305.0, 305.22, 1.4686, 217.67, 596.0, 1.71865], [310.0, 310.24, 1.5546, 221.25, 572.3, 1.73498], [315.0, 315.27, 1.6442, 224.85, 549.8, 1.75106], [320.0, 320.29, 1.7375, 228.42, 528.6, 1.7669], [325.0, 325.31, 1.8345, 232.02, 508.4, 1.78249], [330.0, 330.34, 1.9352, 235.61, 489.4, 1.79783], [340.0, 340.42, 2.149, 242.82, 454.1, 1.8279], [350.0, 350.49, 2.379, 250.02, 422.2, 1.85708], [360.0, 360.58, 2.626, 257.24, 393.4, 1.88543], [370.0, 370.67, 2.892, 264.46, 367.2, 1.91313], [380.0, 380.77, 3.176, 271.69, 343.4, 1.94001], [390.0, 390.88, 3.481, 278.93, 321.5, 1.96633], [400.0, 400.98, 3.806, 286.16, 301.6, 1.99194], [410.0, 411.12, 4.153, 293.43, 283.3, 2.01699], [420.0, 421.26, 4.522, 300.69, 266.6, 2.04142], [430.0, 431.43, 4.915, 307.99, 251.1, 2.06533], [440.0, 441.61, 5.332, 315.3, 236.8, 2.0887], [450.0, 451.8, 5.775, 322.62, 223.6, 2.11161], [460.0, 462.02, 6.245, 329.97, 211.4, 2.13407], [470.0, 472.24, 6.742, 337.32, 200.1, 2.15604], [480.0, 482.49, 7.268, 344.7, 189.5, 2.1776], [490.0, 492.74, 7.824, 352.08, 179.7, 2.19876], [500.0, 503.02, 8.411, 359.49, 170.6, 2.21952], [510.0, 513.32, 9.031, 366.92, 162.1, 2.23993], [520.0, 523.63, 9.684, 374.36, 154.1, 2.25997], [530.0, 533.98, 10.37, 381.84, 146.7, 2.27967], [540.0, 544.35, 11.1, 389.34, 139.7, 2.29906], [550.0, 555.74, 11.86, 396.86, 133.1, 2.31809], [560.0, 565.17, 12.66, 404.42, 127.0, 2.33685], [570.0, 575.59, 13.5, 411.97, 121.2, 2.35531], [580.0, 586.04, 14.38, 419.55, 115.7, 2.37348], [590.0, 596.52, 15.31, 427.15, 110.6, 2.3914], [600.0, 607.02, 16.28, 434.78, 105.8, 2.40902], [610.0, 617.53, 17.3, 442.42, 101.2, 2.42644], [620.0, 628.07, 18.36, 450.09, 96.92, 2.44356], [630.0, 638.63, 19.84, 457.78, 92.84, 2.46048], [640.0, 649.22, 20.64, 465.5, 88.99, 2.47716], [650.0, 659.84, 21.86, 473.25, 85.34, 2.49364], [660.0, 670.47, 23.13, 481.01, 81.89, 2.50985], [670.0, 681.14, 24.46, 488.81, 78.61, 2.52589], [680.0, 691.82, 25.85, 496.62, 75.5, 2.54175], [690.0, 702.52, 27.29, 504.45, 72.56, 2.55731], [700.0, 713.27, 28.8, 512.33, 69.76, 2.57277], [710.0, 724.04, 30.38, 520.23, 67.07, 2.5881], [720.0, 734.82, 32.02, 528.14, 64.53, 2.60319], [730.0, 745.62, 33.72, 536.07, 62.13, 2.61803], [740.0, 756.44, 35.5, 544.02, 59.82, 2.6328], [750.0, 767.29, 37.35, 551.99, 57.63, 2.64737], [760.0, 778.18, 39.27, 560.01, 55.54, 2.66176], [780.0, 800.03, 43.35, 576.12, 51.64, 2.69013], [800.0, 821.95, 47.75, 592.3, 48.08, 2.71787], [820.0, 843.98, 52.59, 608.59, 44.84, 2.74504], [840.0, 866.08, 57.6, 624.95, 41.85, 2.7717], [860.0, 888.27, 63.09, 641.4, 39.12, 2.79783], [880.0, 910.56, 68.98, 657.95, 36.61, 2.82344], [900.0, 932.93, 75.29, 674.58, 34.31, 2.84856], [920.0, 955.38, 82.05, 691.28, 32.18, 2.87324], [940.0, 977.92, 89.28, 708.08, 30.22, 2.89748], [960.0, 1000.55, 97.0, 725.02, 28.4, 2.92128], [980.0, 1023.25, 105.2, 741.98, 26.73, 2.94468], [1000.0, 1046.04, 114.0, 758.94, 25.17, 2.9677], [1020.0, 1068.89, 123.4, 776.1, 23.72, 2.99034], [1040.0, 1091.85, 133.3, 793.36, 23.29, 3.0126], [1060.0, 1114.86, 143.9, 810.62, 21.14, 3.03449], [1080.0, 1137.89, 155.2, 827.88, 19.98, 3.05608], [1100.0, 1161.07, 167.1, 845.33, 18.896, 3.07732], [1120.0, 1184.28, 179.7, 862.79, 17.886, 3.09825], [1140.0, 1207.57, 193.1, 880.35, 16.946, 3.11883], [1160.0, 1230.92, 207.2, 897.91, 16.064, 3.13916], [1180.0, 1254.34, 222.2, 915.57, 15.241, 3.15916], [1200.0, 1277.79, 238.0, 933.33, 14.47, 3.17888], [1220.0, 1301.31, 254.7, 951.09, 13.747, 3.19834], [1240.0, 1324.93, 272.3, 968.95, 13.069, 3.21751], [1260.0, 1348.55, 290.8, 986.9, 12.435, 3.23638], [1280.0, 1372.24, 310.4, 1004.76, 11.835, 3.2551], [1300.0, 1395.97, 330.9, 1022.82, 11.275, 3.27345], [1320.0, 1419.76, 352.5, 1040.88, 10.747, 3.2916], [1340.0, 1443.6, 375.3, 1058.94, 10.247, 3.30959], [1360.0, 1467.49, 399.1, 1077.1, 9.78, 3.32724], [1380.0, 1491.44, 424.2, 1095.26, 9.337, 3.34474], [1400.0, 1515.42, 450.5, 1113.52, 8.919, 3.362], [1420.0, 1539.44, 478.0, 1131.77, 8.526, 3.37901], [1440.0, 1563.51, 506.9, 1150.13, 8.153, 3.39586], [1460.0, 1587.63, 537.1, 1168.49, 7.801, 3.41247], [1480.0, 1611.79, 568.8, 1186.95, 7.468, 3.42892], [1500.0, 1635.97, 601.9, 1205.41, 7.152, 3.44516], [1520.0, 1660.23, 636.5, 1223.87, 6.854, 3.4612], [1540.0, 1684.51, 672.8, 1242.43, 6.569, 3.47712], [1560.0, 1708.82, 710.5, 1260.99, 6.301, 3.49276], [1580.0, 1733.17, 750.0, 1279.65, 6.046, 3.50829], [1600.0, 1757.57, 791.2, 1298.3, 5.804, 3.52364], [1620.0, 1782.0, 834.1, 1316.96, 5.574, 3.53879], [1640.0, 1806.46, 878.9, 1335.72, 5.355, 3.55381], [1660.0, 1830.96, 925.6, 1354.48, 5.147, 3.56867], [1680.0, 1855.5, 974.2, 1373.24, 4.949, 3.58335], [1700.0, 1880.1, 1025.0, 1392.7, 4.761, 3.5979], [1750.0, 1941.6, 1161.0, 1439.8, 4.328, 3.6336], [1800.0, 2003.3, 1310.0, 1487.2, 3.994, 3.6684]])
COL = dict(T=0, h=1, Pr=2, u=3, vr=4, s0=5)
def tab(col, T):                             # value of a column at temperature T (linear interpolation, as a student does)
    return float(np.interp(T, A17[:, 0], A17[:, COL[col]]))
def inv(col, val):                           # temperature at which a column has the value val
    x = A17[:, COL[col]]; T = A17[:, 0]
    return float(np.interp(val, x, T)) if x[0] < x[-1] else float(np.interp(val, x[::-1], T[::-1]))
rp, T1, T3, ec, et, eps = 8.0, 300.0, 1300.0, 0.80, 0.85, 0.80
h1 = tab('h', T1); T2s = inv('Pr', rp * tab('Pr', T1)); h3 = tab('h', T3); T4s = inv('Pr', tab('Pr', T3) / rp)
wc = (tab('h', T2s) - h1) / ec; wt = et * (h3 - tab('h', T4s))     # isentropic efficiencies
h2 = h1 + wc; h4 = h3 - wt; wnet = wt - wc
print(f"w_comp = {wc:.2f}, w_turb = {wt:.2f} kJ/kg, r_bw = {wc/wt:.3f}, T2a = {inv('h', h2):.0f} K, T4a = {inv('h', h4):.0f} K")
print(f"without regenerator: q_in = {h3-h2:.2f}, eta = {wnet/(h3-h2):.3f}")
h5 = h2 + eps * (h4 - h2)                                          # effectiveness: (h5 - h2)/(h4 - h2)
print(f"with regenerator: h5 = {h5:.2f}, q_in = {h3-h5:.2f}, eta = {wnet/(h3-h5):.3f}")

w_comp = 304.99, w_turb = 515.60 kJ/kg, r_bw = 0.592, T2a = 598 K, T4a = 853 K
without regenerator: q_in = 790.79, eta = 0.266
with regenerator: h5 = 825.33, q_in = 570.64, eta = 0.369


**Đọc kết quả:** máy nén thực ăn 59.2 % công tuabin (lý tưởng 40.2 %); η rơi từ 0.426 xuống 0.266; T₄ₐ = 853 K cao hơn T₂ₐ = 598 K nên hồi nhiệt có ích: h₅ = 825.33 và η = 0.369 (sách: 0,592; 26,6 %; 825,37; 36,9 %).

## 5. r_p cho công cực đại: công thức, tối ưu số và nhiệt dung thay đổi (slide tr.40)

**Câu hỏi:** với T_min = 300 K, T_max = 1000 K, r_p = 8,2 của slide đúng với giả thiết nào, và công thực phẳng cỡ nào quanh cực đại?

In [5]:
import math
from scipy.integrate import quad
from scipy.optimize import brentq, minimize_scalar
k, cp, R = 1.4, 1.005, 0.287
T1, T3 = 300.0, 1000.0
wnet_k = lambda rp: cp*(T3 - T1*rp**((k-1)/k)) - cp*(T3/rp**((k-1)/k) - T1)      # w_turb - w_comp
r_formula = (T3/T1)**(k/(2*(k-1)))
r_num = minimize_scalar(lambda x: -wnet_k(x), bounds=(2, 40), method='bounded', options=dict(xatol=1e-9)).x
print(f"formula {r_formula:.3f}; numerical {r_num:.3f}; w_net,max = {wnet_k(r_num):.1f} kJ/kg; w(8.2)/w_max = {wnet_k(8.2)/wnet_k(r_num):.4f}")
# variable cp: polynomial of Table A-2c
a, b, c, d, M = 28.11, 0.001967, 4.802e-6, -1.966e-9, 28.97
cpT = lambda T: (a + b*T + c*T**2 + d*T**3) / M
h = lambda x, y: quad(cpT, x, y)[0]
phi = lambda x, y: quad(lambda T: cpT(T)/T, x, y)[0]
def wnet_v(rp):
    T2 = brentq(lambda T: phi(T1, T) - R*math.log(rp), T1, 1000); T4 = brentq(lambda T: phi(T, T3) - R*math.log(rp), 300, T3)
    return h(T4, T3) - h(T1, T2)
r_v = minimize_scalar(lambda x: -wnet_v(x), bounds=(3, 30), method='bounded', options=dict(xatol=1e-6)).x
print(f"variable c_p (polynomial): r_p,opt = {r_v:.2f}; w(8.2)/w_max = {wnet_v(8.2)/wnet_v(r_v):.4f}")
for rp in (2, 8.2, 15):
    print(f"r_p = {rp:5.1f}: eta = {1 - rp**(-(k-1)/k):.3f}, w_net = {wnet_k(rp):6.1f} kJ/kg")

formula 8.223; numerical 8.223; w_net,max = 205.6 kJ/kg; w(8.2)/w_max = 1.0000
variable c_p (polynomial): r_p,opt = 9.13; w(8.2)/w_max = 0.9977
r_p =   2.0: eta = 0.180, w_net =  114.5 kJ/kg
r_p =   8.2: eta = 0.452, w_net =  205.6 kJ/kg
r_p =  15.0: eta = 0.539, w_net =  189.3 kJ/kg


**Đọc kết quả:** r_p = 8,2 của slide là nghiệm của k = 1,4 hằng (8.223); với c_p(T) cực đại dịch lên khoảng 9,1 nhưng công thực rất phẳng (tỉ số 0,998), nên 8,2 vẫn gần tối ưu. Ở r_p = 8,2: η = 0.452, w_net = 205.6 kJ/kg; hiệu suất cực đại và công cực đại không trùng nhau.

## 6. Hai cấp nén và giãn, làm mát trung gian, hâm nóng, hồi nhiệt (Cengel Ví dụ 9–8)

**Câu hỏi:** r_p = 8 chia hai cấp (√8 mỗi cấp), 300 K và 1300 K: r_bw và η không hồi nhiệt và với hồi nhiệt lý tưởng khác chu trình một cấp thế nào?

In [6]:
import math, numpy as np
from scipy.integrate import quad
from scipy.optimize import brentq, minimize_scalar
R = 0.287                                    # kJ/(kg K), Table A-1
# Table A-17 (air, ideal gas), rows read from the page image: T [K], h [kJ/kg], P_r, u [kJ/kg], v_r, s0
A17 = np.array([[200.0, 199.97, 0.3363, 142.56, 1707.0, 1.29559], [210.0, 209.97, 0.3987, 149.69, 1512.0, 1.34444], [220.0, 219.97, 0.469, 156.82, 1346.0, 1.39105], [230.0, 230.02, 0.5477, 164.0, 1205.0, 1.43557], [240.0, 240.02, 0.6355, 171.13, 1084.0, 1.47824], [250.0, 250.05, 0.7329, 178.28, 979.0, 1.51917], [260.0, 260.09, 0.8405, 185.45, 887.8, 1.55848], [270.0, 270.11, 0.959, 192.6, 808.0, 1.59634], [280.0, 280.13, 1.0889, 199.75, 738.0, 1.63279], [285.0, 285.14, 1.1584, 203.33, 706.1, 1.65055], [290.0, 290.16, 1.2311, 206.91, 676.1, 1.66802], [295.0, 295.17, 1.3068, 210.49, 647.9, 1.68515], [298.0, 298.18, 1.3543, 212.64, 631.9, 1.69528], [300.0, 300.19, 1.386, 214.07, 621.2, 1.70203], [305.0, 305.22, 1.4686, 217.67, 596.0, 1.71865], [310.0, 310.24, 1.5546, 221.25, 572.3, 1.73498], [315.0, 315.27, 1.6442, 224.85, 549.8, 1.75106], [320.0, 320.29, 1.7375, 228.42, 528.6, 1.7669], [325.0, 325.31, 1.8345, 232.02, 508.4, 1.78249], [330.0, 330.34, 1.9352, 235.61, 489.4, 1.79783], [340.0, 340.42, 2.149, 242.82, 454.1, 1.8279], [350.0, 350.49, 2.379, 250.02, 422.2, 1.85708], [360.0, 360.58, 2.626, 257.24, 393.4, 1.88543], [370.0, 370.67, 2.892, 264.46, 367.2, 1.91313], [380.0, 380.77, 3.176, 271.69, 343.4, 1.94001], [390.0, 390.88, 3.481, 278.93, 321.5, 1.96633], [400.0, 400.98, 3.806, 286.16, 301.6, 1.99194], [410.0, 411.12, 4.153, 293.43, 283.3, 2.01699], [420.0, 421.26, 4.522, 300.69, 266.6, 2.04142], [430.0, 431.43, 4.915, 307.99, 251.1, 2.06533], [440.0, 441.61, 5.332, 315.3, 236.8, 2.0887], [450.0, 451.8, 5.775, 322.62, 223.6, 2.11161], [460.0, 462.02, 6.245, 329.97, 211.4, 2.13407], [470.0, 472.24, 6.742, 337.32, 200.1, 2.15604], [480.0, 482.49, 7.268, 344.7, 189.5, 2.1776], [490.0, 492.74, 7.824, 352.08, 179.7, 2.19876], [500.0, 503.02, 8.411, 359.49, 170.6, 2.21952], [510.0, 513.32, 9.031, 366.92, 162.1, 2.23993], [520.0, 523.63, 9.684, 374.36, 154.1, 2.25997], [530.0, 533.98, 10.37, 381.84, 146.7, 2.27967], [540.0, 544.35, 11.1, 389.34, 139.7, 2.29906], [550.0, 555.74, 11.86, 396.86, 133.1, 2.31809], [560.0, 565.17, 12.66, 404.42, 127.0, 2.33685], [570.0, 575.59, 13.5, 411.97, 121.2, 2.35531], [580.0, 586.04, 14.38, 419.55, 115.7, 2.37348], [590.0, 596.52, 15.31, 427.15, 110.6, 2.3914], [600.0, 607.02, 16.28, 434.78, 105.8, 2.40902], [610.0, 617.53, 17.3, 442.42, 101.2, 2.42644], [620.0, 628.07, 18.36, 450.09, 96.92, 2.44356], [630.0, 638.63, 19.84, 457.78, 92.84, 2.46048], [640.0, 649.22, 20.64, 465.5, 88.99, 2.47716], [650.0, 659.84, 21.86, 473.25, 85.34, 2.49364], [660.0, 670.47, 23.13, 481.01, 81.89, 2.50985], [670.0, 681.14, 24.46, 488.81, 78.61, 2.52589], [680.0, 691.82, 25.85, 496.62, 75.5, 2.54175], [690.0, 702.52, 27.29, 504.45, 72.56, 2.55731], [700.0, 713.27, 28.8, 512.33, 69.76, 2.57277], [710.0, 724.04, 30.38, 520.23, 67.07, 2.5881], [720.0, 734.82, 32.02, 528.14, 64.53, 2.60319], [730.0, 745.62, 33.72, 536.07, 62.13, 2.61803], [740.0, 756.44, 35.5, 544.02, 59.82, 2.6328], [750.0, 767.29, 37.35, 551.99, 57.63, 2.64737], [760.0, 778.18, 39.27, 560.01, 55.54, 2.66176], [780.0, 800.03, 43.35, 576.12, 51.64, 2.69013], [800.0, 821.95, 47.75, 592.3, 48.08, 2.71787], [820.0, 843.98, 52.59, 608.59, 44.84, 2.74504], [840.0, 866.08, 57.6, 624.95, 41.85, 2.7717], [860.0, 888.27, 63.09, 641.4, 39.12, 2.79783], [880.0, 910.56, 68.98, 657.95, 36.61, 2.82344], [900.0, 932.93, 75.29, 674.58, 34.31, 2.84856], [920.0, 955.38, 82.05, 691.28, 32.18, 2.87324], [940.0, 977.92, 89.28, 708.08, 30.22, 2.89748], [960.0, 1000.55, 97.0, 725.02, 28.4, 2.92128], [980.0, 1023.25, 105.2, 741.98, 26.73, 2.94468], [1000.0, 1046.04, 114.0, 758.94, 25.17, 2.9677], [1020.0, 1068.89, 123.4, 776.1, 23.72, 2.99034], [1040.0, 1091.85, 133.3, 793.36, 23.29, 3.0126], [1060.0, 1114.86, 143.9, 810.62, 21.14, 3.03449], [1080.0, 1137.89, 155.2, 827.88, 19.98, 3.05608], [1100.0, 1161.07, 167.1, 845.33, 18.896, 3.07732], [1120.0, 1184.28, 179.7, 862.79, 17.886, 3.09825], [1140.0, 1207.57, 193.1, 880.35, 16.946, 3.11883], [1160.0, 1230.92, 207.2, 897.91, 16.064, 3.13916], [1180.0, 1254.34, 222.2, 915.57, 15.241, 3.15916], [1200.0, 1277.79, 238.0, 933.33, 14.47, 3.17888], [1220.0, 1301.31, 254.7, 951.09, 13.747, 3.19834], [1240.0, 1324.93, 272.3, 968.95, 13.069, 3.21751], [1260.0, 1348.55, 290.8, 986.9, 12.435, 3.23638], [1280.0, 1372.24, 310.4, 1004.76, 11.835, 3.2551], [1300.0, 1395.97, 330.9, 1022.82, 11.275, 3.27345], [1320.0, 1419.76, 352.5, 1040.88, 10.747, 3.2916], [1340.0, 1443.6, 375.3, 1058.94, 10.247, 3.30959], [1360.0, 1467.49, 399.1, 1077.1, 9.78, 3.32724], [1380.0, 1491.44, 424.2, 1095.26, 9.337, 3.34474], [1400.0, 1515.42, 450.5, 1113.52, 8.919, 3.362], [1420.0, 1539.44, 478.0, 1131.77, 8.526, 3.37901], [1440.0, 1563.51, 506.9, 1150.13, 8.153, 3.39586], [1460.0, 1587.63, 537.1, 1168.49, 7.801, 3.41247], [1480.0, 1611.79, 568.8, 1186.95, 7.468, 3.42892], [1500.0, 1635.97, 601.9, 1205.41, 7.152, 3.44516], [1520.0, 1660.23, 636.5, 1223.87, 6.854, 3.4612], [1540.0, 1684.51, 672.8, 1242.43, 6.569, 3.47712], [1560.0, 1708.82, 710.5, 1260.99, 6.301, 3.49276], [1580.0, 1733.17, 750.0, 1279.65, 6.046, 3.50829], [1600.0, 1757.57, 791.2, 1298.3, 5.804, 3.52364], [1620.0, 1782.0, 834.1, 1316.96, 5.574, 3.53879], [1640.0, 1806.46, 878.9, 1335.72, 5.355, 3.55381], [1660.0, 1830.96, 925.6, 1354.48, 5.147, 3.56867], [1680.0, 1855.5, 974.2, 1373.24, 4.949, 3.58335], [1700.0, 1880.1, 1025.0, 1392.7, 4.761, 3.5979], [1750.0, 1941.6, 1161.0, 1439.8, 4.328, 3.6336], [1800.0, 2003.3, 1310.0, 1487.2, 3.994, 3.6684]])
COL = dict(T=0, h=1, Pr=2, u=3, vr=4, s0=5)
def tab(col, T):                             # value of a column at temperature T (linear interpolation, as a student does)
    return float(np.interp(T, A17[:, 0], A17[:, COL[col]]))
def inv(col, val):                           # temperature at which a column has the value val
    x = A17[:, COL[col]]; T = A17[:, 0]
    return float(np.interp(val, x, T)) if x[0] < x[-1] else float(np.interp(val, x[::-1], T[::-1]))
rp, T1, T3 = 8.0, 300.0, 1300.0; x = math.sqrt(rp)
h1 = tab('h', T1); T2 = inv('Pr', x * tab('Pr', T1)); h2 = tab('h', T2)
h6 = tab('h', T3); T7 = inv('Pr', tab('Pr', T3) / x); h7 = tab('h', T7)
wc = 2 * (h2 - h1); wt = 2 * (h6 - h7); wnet = wt - wc
q_noreg = (h6 - h2) + (h6 - h7)           # h4 = h2 (intercooled stages), reheat to T3 after the first turbine
q_idealreg = (h6 - h7) + (h6 - h7)        # ideal regenerator: h5 = h7
print(f"T2 = {T2:.1f} K, T7 = {T7:.1f} K; w_comp = {wc:.2f}, w_turb = {wt:.2f}, w_net = {wnet:.2f} kJ/kg, r_bw = {wc/wt:.3f}")
print(f"no regenerator: q_in = {q_noreg:.2f}, eta = {wnet/q_noreg:.3f}; ideal regenerator: q_in = {q_idealreg:.2f}, eta = {wnet/q_idealreg:.3f}")

T2 = 403.3 K, T7 = 1006.4 K; w_comp = 208.25, w_turb = 685.32, w_net = 477.07 kJ/kg, r_bw = 0.304
no regenerator: q_in = 1334.31, eta = 0.358; ideal regenerator: q_in = 685.32, eta = 0.696


**Đọc kết quả:** r_bw giảm xuống 0.304 (một cấp 0.402) nhưng η không hồi nhiệt cũng giảm xuống 0.358 (một cấp 0.426) vì nhiệt vào tăng; thêm hồi nhiệt lý tưởng đưa η lên 0.696 (sách 0,304; 35,8 %; 69,6 %).

## 7. Chu trình phản lực lý tưởng: Cengel Ví dụ 9–9 và El-Sayed Ví dụ 4.2

**Câu hỏi:** tính chuỗi khuếch tán → máy nén → buồng đốt → tuabin → vòi phun của hai ví dụ; kết quả in của El-Sayed có khớp không?

In [7]:
import math
k = 1.4
# Cengel Example 9-9 (English units, cold air): 850 ft/s, 5 psia, 420 R, rp = 10, T4 = 2460 R, 100 lbm/s, cp = 0.240 Btu/lbm R, 1 Btu/lbm = 25037 ft2/s2
cp, J = 0.240, 25037.0
T1, P1, V, rp, T4, md = 420.0, 5.0, 850.0, 10.0, 2460.0, 100.0
T2 = T1 + V**2 / (2*cp*J); P2 = P1 * (T2/T1)**(k/(k-1))
T3 = T2 * rp**((k-1)/k); P3 = rp * P2
T5 = T4 - T3 + T2; P5 = P3 * (T5/T4)**(k/(k-1))                 # turbine work = compressor work
T6 = T5 * (P1/P5)**((k-1)/k); V6 = math.sqrt(2*J*cp*(T5 - T6))
Wp = md * (V6 - V) * V / J; Qin = md * cp * (T4 - T3)
print(f"9-9: T2 = {T2:.0f} R, T3 = {T3:.0f} R, T5 = {T5:.0f} R, P5 = {P5:.1f} psia, V6 = {V6:.0f} ft/s, Wp = {Wp:.0f}, Qin = {Qin:.0f} Btu/s, eta_P = {100*Wp/Qin:.1f} %")
# El-Sayed Example 4.2 (SI): 310 m/s, 240 K, 32 kPa, pi_c = 12, T04 = 1400 K, QR = 42700 kJ/kg, cp = 1.005
cp, Ta, Pa, Vf, pic, T04, QR = 1.005, 240.0, 32.0, 310.0, 12.0, 1400.0, 42700.0
T02 = Ta + Vf**2 / (2000*cp); P02 = Pa * (T02/Ta)**3.5
T03 = T02 * pic**((k-1)/k); f = cp*(T04 - T03) / (QR - cp*T04)
T05 = T04 - (T03 - T02)/(1 + f); P05 = pic*P02*(T05/T04)**3.5
T7 = T05 * (Pa/P05)**((k-1)/k); Vj = math.sqrt(2000*cp*(T05 - T7))
st = (1 + f)*Vj - Vf; eta_p = 2*Vf/(Vf + Vj)
print(f"4.2: T03 = {T03:.1f} K, f = {f:.4f}, T05 = {T05:.1f} K, T7 = {T7:.1f} K, Vj = {Vj:.1f} m/s, specific thrust = {st:.1f} N s/kg, m_a = {12120/st:.2f} kg/s")
# the book's own numbers: T03 = 587.5 K and T7 = 563.6 K (slips). Vj from the book's T05 and T7:
print(f"book: T7 = 563.6 K gives Vj = {math.sqrt(2000*cp*(1106.1 - 563.6)):.1f} m/s; correct T7 = 1106.1/1.9283 = {1106.1/1.9283:.1f} K; 287.8 x 2.035 = {287.8*2.035:.1f} K")

9-9: T2 = 480 R, T3 = 927 R, T5 = 2013 R, P5 = 39.6 psia, V6 = 3286 ft/s, Wp = 8271, Qin = 36793 Btu/s, eta_P = 22.5 %
4.2: T03 = 585.4 K, f = 0.0198, T05 = 1108.2 K, T7 = 574.0 K, Vj = 1036.2 m/s, specific thrust = 746.8 N s/kg, m_a = 16.23 kg/s
book: T7 = 563.6 K gives Vj = 1044.2 m/s; correct T7 = 1106.1/1.9283 = 573.6 K; 287.8 x 2.035 = 585.7 K


**Đọc kết quả:** Ví dụ 9–9 khớp sách (480; 927; 2013 R; 39,7 psia; 3288 ft/s; 22,5 %): V₆ = 3286 ft/s, η_P = 22.5 %. Ví dụ 4.2: V_j = 1036.2 m/s, ṁ_a = 16.23 kg/s, không phải 1044,34 m/s và 16,06 kg/s của sách: sách tính sai T₀₃ (587,5 thay vì 585,7 K) và T₇ (563,6 thay vì 573,6 K); với T₇ = 563,6 K đúng ra 1044 m/s. Ghi trong mục lỗi nguồn.

## 8. Otto và Diesel theo r, và bài Tutorials I #8 không có lời giải

**Câu hỏi:** (a) với k = 1,4, Otto hơn Diesel (r_c = 2) bao nhiêu ở các r điển hình, và hình slide tr.25 lệch công thức bao nhiêu? (b) giải Tutorials I #8 bằng bảng A-17.

In [8]:
import math, numpy as np
from scipy.integrate import quad
from scipy.optimize import brentq, minimize_scalar
R = 0.287                                    # kJ/(kg K), Table A-1
# Table A-17 (air, ideal gas), rows read from the page image: T [K], h [kJ/kg], P_r, u [kJ/kg], v_r, s0
A17 = np.array([[200.0, 199.97, 0.3363, 142.56, 1707.0, 1.29559], [210.0, 209.97, 0.3987, 149.69, 1512.0, 1.34444], [220.0, 219.97, 0.469, 156.82, 1346.0, 1.39105], [230.0, 230.02, 0.5477, 164.0, 1205.0, 1.43557], [240.0, 240.02, 0.6355, 171.13, 1084.0, 1.47824], [250.0, 250.05, 0.7329, 178.28, 979.0, 1.51917], [260.0, 260.09, 0.8405, 185.45, 887.8, 1.55848], [270.0, 270.11, 0.959, 192.6, 808.0, 1.59634], [280.0, 280.13, 1.0889, 199.75, 738.0, 1.63279], [285.0, 285.14, 1.1584, 203.33, 706.1, 1.65055], [290.0, 290.16, 1.2311, 206.91, 676.1, 1.66802], [295.0, 295.17, 1.3068, 210.49, 647.9, 1.68515], [298.0, 298.18, 1.3543, 212.64, 631.9, 1.69528], [300.0, 300.19, 1.386, 214.07, 621.2, 1.70203], [305.0, 305.22, 1.4686, 217.67, 596.0, 1.71865], [310.0, 310.24, 1.5546, 221.25, 572.3, 1.73498], [315.0, 315.27, 1.6442, 224.85, 549.8, 1.75106], [320.0, 320.29, 1.7375, 228.42, 528.6, 1.7669], [325.0, 325.31, 1.8345, 232.02, 508.4, 1.78249], [330.0, 330.34, 1.9352, 235.61, 489.4, 1.79783], [340.0, 340.42, 2.149, 242.82, 454.1, 1.8279], [350.0, 350.49, 2.379, 250.02, 422.2, 1.85708], [360.0, 360.58, 2.626, 257.24, 393.4, 1.88543], [370.0, 370.67, 2.892, 264.46, 367.2, 1.91313], [380.0, 380.77, 3.176, 271.69, 343.4, 1.94001], [390.0, 390.88, 3.481, 278.93, 321.5, 1.96633], [400.0, 400.98, 3.806, 286.16, 301.6, 1.99194], [410.0, 411.12, 4.153, 293.43, 283.3, 2.01699], [420.0, 421.26, 4.522, 300.69, 266.6, 2.04142], [430.0, 431.43, 4.915, 307.99, 251.1, 2.06533], [440.0, 441.61, 5.332, 315.3, 236.8, 2.0887], [450.0, 451.8, 5.775, 322.62, 223.6, 2.11161], [460.0, 462.02, 6.245, 329.97, 211.4, 2.13407], [470.0, 472.24, 6.742, 337.32, 200.1, 2.15604], [480.0, 482.49, 7.268, 344.7, 189.5, 2.1776], [490.0, 492.74, 7.824, 352.08, 179.7, 2.19876], [500.0, 503.02, 8.411, 359.49, 170.6, 2.21952], [510.0, 513.32, 9.031, 366.92, 162.1, 2.23993], [520.0, 523.63, 9.684, 374.36, 154.1, 2.25997], [530.0, 533.98, 10.37, 381.84, 146.7, 2.27967], [540.0, 544.35, 11.1, 389.34, 139.7, 2.29906], [550.0, 555.74, 11.86, 396.86, 133.1, 2.31809], [560.0, 565.17, 12.66, 404.42, 127.0, 2.33685], [570.0, 575.59, 13.5, 411.97, 121.2, 2.35531], [580.0, 586.04, 14.38, 419.55, 115.7, 2.37348], [590.0, 596.52, 15.31, 427.15, 110.6, 2.3914], [600.0, 607.02, 16.28, 434.78, 105.8, 2.40902], [610.0, 617.53, 17.3, 442.42, 101.2, 2.42644], [620.0, 628.07, 18.36, 450.09, 96.92, 2.44356], [630.0, 638.63, 19.84, 457.78, 92.84, 2.46048], [640.0, 649.22, 20.64, 465.5, 88.99, 2.47716], [650.0, 659.84, 21.86, 473.25, 85.34, 2.49364], [660.0, 670.47, 23.13, 481.01, 81.89, 2.50985], [670.0, 681.14, 24.46, 488.81, 78.61, 2.52589], [680.0, 691.82, 25.85, 496.62, 75.5, 2.54175], [690.0, 702.52, 27.29, 504.45, 72.56, 2.55731], [700.0, 713.27, 28.8, 512.33, 69.76, 2.57277], [710.0, 724.04, 30.38, 520.23, 67.07, 2.5881], [720.0, 734.82, 32.02, 528.14, 64.53, 2.60319], [730.0, 745.62, 33.72, 536.07, 62.13, 2.61803], [740.0, 756.44, 35.5, 544.02, 59.82, 2.6328], [750.0, 767.29, 37.35, 551.99, 57.63, 2.64737], [760.0, 778.18, 39.27, 560.01, 55.54, 2.66176], [780.0, 800.03, 43.35, 576.12, 51.64, 2.69013], [800.0, 821.95, 47.75, 592.3, 48.08, 2.71787], [820.0, 843.98, 52.59, 608.59, 44.84, 2.74504], [840.0, 866.08, 57.6, 624.95, 41.85, 2.7717], [860.0, 888.27, 63.09, 641.4, 39.12, 2.79783], [880.0, 910.56, 68.98, 657.95, 36.61, 2.82344], [900.0, 932.93, 75.29, 674.58, 34.31, 2.84856], [920.0, 955.38, 82.05, 691.28, 32.18, 2.87324], [940.0, 977.92, 89.28, 708.08, 30.22, 2.89748], [960.0, 1000.55, 97.0, 725.02, 28.4, 2.92128], [980.0, 1023.25, 105.2, 741.98, 26.73, 2.94468], [1000.0, 1046.04, 114.0, 758.94, 25.17, 2.9677], [1020.0, 1068.89, 123.4, 776.1, 23.72, 2.99034], [1040.0, 1091.85, 133.3, 793.36, 23.29, 3.0126], [1060.0, 1114.86, 143.9, 810.62, 21.14, 3.03449], [1080.0, 1137.89, 155.2, 827.88, 19.98, 3.05608], [1100.0, 1161.07, 167.1, 845.33, 18.896, 3.07732], [1120.0, 1184.28, 179.7, 862.79, 17.886, 3.09825], [1140.0, 1207.57, 193.1, 880.35, 16.946, 3.11883], [1160.0, 1230.92, 207.2, 897.91, 16.064, 3.13916], [1180.0, 1254.34, 222.2, 915.57, 15.241, 3.15916], [1200.0, 1277.79, 238.0, 933.33, 14.47, 3.17888], [1220.0, 1301.31, 254.7, 951.09, 13.747, 3.19834], [1240.0, 1324.93, 272.3, 968.95, 13.069, 3.21751], [1260.0, 1348.55, 290.8, 986.9, 12.435, 3.23638], [1280.0, 1372.24, 310.4, 1004.76, 11.835, 3.2551], [1300.0, 1395.97, 330.9, 1022.82, 11.275, 3.27345], [1320.0, 1419.76, 352.5, 1040.88, 10.747, 3.2916], [1340.0, 1443.6, 375.3, 1058.94, 10.247, 3.30959], [1360.0, 1467.49, 399.1, 1077.1, 9.78, 3.32724], [1380.0, 1491.44, 424.2, 1095.26, 9.337, 3.34474], [1400.0, 1515.42, 450.5, 1113.52, 8.919, 3.362], [1420.0, 1539.44, 478.0, 1131.77, 8.526, 3.37901], [1440.0, 1563.51, 506.9, 1150.13, 8.153, 3.39586], [1460.0, 1587.63, 537.1, 1168.49, 7.801, 3.41247], [1480.0, 1611.79, 568.8, 1186.95, 7.468, 3.42892], [1500.0, 1635.97, 601.9, 1205.41, 7.152, 3.44516], [1520.0, 1660.23, 636.5, 1223.87, 6.854, 3.4612], [1540.0, 1684.51, 672.8, 1242.43, 6.569, 3.47712], [1560.0, 1708.82, 710.5, 1260.99, 6.301, 3.49276], [1580.0, 1733.17, 750.0, 1279.65, 6.046, 3.50829], [1600.0, 1757.57, 791.2, 1298.3, 5.804, 3.52364], [1620.0, 1782.0, 834.1, 1316.96, 5.574, 3.53879], [1640.0, 1806.46, 878.9, 1335.72, 5.355, 3.55381], [1660.0, 1830.96, 925.6, 1354.48, 5.147, 3.56867], [1680.0, 1855.5, 974.2, 1373.24, 4.949, 3.58335], [1700.0, 1880.1, 1025.0, 1392.7, 4.761, 3.5979], [1750.0, 1941.6, 1161.0, 1439.8, 4.328, 3.6336], [1800.0, 2003.3, 1310.0, 1487.2, 3.994, 3.6684]])
COL = dict(T=0, h=1, Pr=2, u=3, vr=4, s0=5)
def tab(col, T):                             # value of a column at temperature T (linear interpolation, as a student does)
    return float(np.interp(T, A17[:, 0], A17[:, COL[col]]))
def inv(col, val):                           # temperature at which a column has the value val
    x = A17[:, COL[col]]; T = A17[:, 0]
    return float(np.interp(val, x, T)) if x[0] < x[-1] else float(np.interp(val, x[::-1], T[::-1]))
k, rc = 1.4, 2.0
print("   r   Otto  Diesel(rc=2)  gap")
for r in (7, 10, 14, 16, 18, 24):
    eo = 1 - r**(1-k); ed = 1 - r**(1-k) * (rc**k - 1) / (k*(rc - 1))
    print(f"{r:4d}  {eo:.3f}  {ed:.3f}        {eo-ed:.3f}")
print(f"slide p.25 reads about 0.665 at r = 14; formula {1-14**(1-k):.3f}")
# Tutorials I #8: 300 K, 100 kPa -> 700 kPa and 580 K (actual state), q_in = 950, eta_t = 0.86, P3 = P2, exhaust at 100 kPa
rp = 7.0; h1 = tab('h', 300); h2 = tab('h', 580); wc = h2 - h1
h3 = h2 + 950; T3 = inv('h', h3); T4s = inv('Pr', tab('Pr', T3) / rp); wt = 0.86 * (h3 - tab('h', T4s)); wnet = wt - wc
print(f"#8: T3 = {T3:.0f} K, w_turb = {wt:.1f}, w_comp = {wc:.1f} kJ/kg, w_comp/w_turb = {wc/wt:.3f}, eta = {wnet/950:.3f}")

   r   Otto  Diesel(rc=2)  gap
   7  0.541  0.462        0.078
  10  0.602  0.534        0.068
  14  0.652  0.593        0.059
  16  0.670  0.614        0.056
  18  0.685  0.632        0.054
  24  0.720  0.672        0.048
slide p.25 reads about 0.665 at r = 14; formula 0.652
#8: T3 = 1417 K, w_turb = 542.0, w_comp = 285.8 kJ/kg, w_comp/w_turb = 0.527, eta = 0.270


**Đọc kết quả:** Otto luôn hơn Diesel cùng r (ví dụ r = 16: 0.670 so với 0.614); công thức cho 0.652 tại r = 14 còn hình slide ≈ 0,665 (lệch 0,013, trong sai số đọc hình). Tutorials I #8: T₃ = 1417 K, w_turb = 542.0, w_comp = 285.8 kJ/kg, tỉ số 0.527, η = 0.270 (giả định P₃ = P₂ và xả ở 100 kPa).